[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/26_recommender_systems/04_ranking_metrics.ipynb)

# Bài 04 — Đánh giá hệ gợi ý: metric xếp hạng và hơn thế nữa

**🎯 Mục tiêu**
- Metric xếp hạng: precision@k, recall@k, hit rate, MRR, MAP@k, **NDCG@k** — tính tay và bằng code.
- Vì sao **RMSE tốt ≠ gợi ý tốt**; dữ liệu thiếu **không ngẫu nhiên** (MNAR) và thiên lệch chọn lựa.
- Ngoài độ chính xác: **độ phủ**, **độ mới lạ**, **độ đa dạng**; re-ranking đa dạng hoá (MMR).
- Đánh giá offline vs online (A/B test), vòng phản hồi; template cho hệ gợi ý.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/26_recommender_systems

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course.datasets import make_ratings
from ml_course.metrics import average_precision_at_k, ndcg_at_k, precision_at_k, recall_at_k
from ml_course.visualization import set_style

set_style()

## 1. Metric xếp hạng — tính tay

Danh sách gợi ý (theo thứ tự) `[A, B, C, D, E]`; user thực sự thích `B` (rất thích, mức 2), `E` (mức 1) và `F` (mức 1, không có trong danh sách).

| Metric | Định nghĩa | Ở đây |
|---|---|---|
| precision@5 | số item đúng trong top-5 / 5 | 2/5 = 0.4 |
| recall@5 | số item đúng trong top-5 / số item đúng | 2/3 |
| hit rate@5 | có ít nhất 1 item đúng? | 1 |
| MRR | 1 / vị trí của item đúng **đầu tiên** | 1/2 |
| AP@5 | trung bình precision tại mỗi vị trí có item đúng (chia min(\|rel\|, k)) | (1/2 + 2/5) / 3 = 0.3 |
| NDCG@5 | DCG / DCG lý tưởng, $\text{DCG} = \sum_i \frac{2^{rel_i} - 1}{\log_2(i + 1)}$ | tính dưới đây |

NDCG thưởng cho việc đặt item **liên quan nhất** ở **đầu** danh sách — người dùng hiếm khi cuộn xuống.

In [ ]:
recs = ["A", "B", "C", "D", "E"]
relevance = {"B": 2, "E": 1, "F": 1}
dcg = (2**2 - 1) / np.log2(3) + (2**1 - 1) / np.log2(6)                 # B ở vị trí 2, E ở vị trí 5
idcg = (2**2 - 1) / np.log2(2) + (2**1 - 1) / np.log2(3) + (2**1 - 1) / np.log2(4)   # B, E, F xếp đầu
first_hit = next(i for i, r in enumerate(recs, 1) if r in relevance)
print(f"precision@5 = {precision_at_k(recs, relevance, 5):.3f} | recall@5 = {recall_at_k(recs, relevance, 5):.3f} | MRR = {1 / first_hit:.3f}")
print(f"AP@5 = {average_precision_at_k(recs, relevance, 5):.3f} | NDCG@5 tính tay = {dcg / idcg:.4f} | ml_course = {ndcg_at_k(recs, relevance, 5):.4f}")
print(f"đổi chỗ: đưa B lên đầu → NDCG@5 = {ndcg_at_k(['B', 'A', 'C', 'D', 'E'], relevance, 5):.4f}")

## 2. RMSE tốt ≠ gợi ý tốt

So sánh các mô hình của bài 01–03 trên **cả hai** loại metric (cùng cách chia dữ liệu):

In [ ]:
ratings = pd.read_csv("../../data/sample/ratings.csv")
items = pd.read_csv("../../data/sample/items.csv")
n_users, n_items = ratings.user_id.max() + 1, ratings.item_id.max() + 1


def split_per_user(ratings, test_frac=0.2, min_ratings=5, seed=0):
    rng = np.random.default_rng(seed)
    test_mask = np.zeros(len(ratings), bool)
    for _, g in ratings.groupby("user_id"):
        if len(g) >= min_ratings:
            test_mask[rng.choice(g.index, max(1, int(test_frac * len(g))), replace=False)] = True
    return ratings[~test_mask], ratings[test_mask]


train, test = split_per_user(ratings)
u_tr, i_tr, y_tr = train.user_id.to_numpy(), train.item_id.to_numpy(), train.rating.to_numpy(float)
u_te, i_te, y_te = test.user_id.to_numpy(), test.item_id.to_numpy(), test.rating.to_numpy(float)
mu = y_tr.mean()
B = np.zeros((n_users, n_items))
B[u_tr, i_tr] = 1.0
popularity = B.sum(0)
seen = {u: set(g.item_id) for u, g in train.groupby("user_id")}
test_ratings = {u: dict(zip(g.item_id, g.rating)) for u, g in test.groupby("user_id")}
genre = items.genre.to_numpy()
p_item = (popularity + 1) / (popularity + 1).sum()


def recommend(score_fn, u, k=10):
    scores = np.asarray(score_fn(u), float).copy()
    scores[list(seen.get(u, []))] = -np.inf
    return np.argsort(-scores)[:k]


def evaluate(score_fn, k=10):
    out = {"precision@10": [], "NDCG@10": [], "novelty": [], "diversity": []}
    recommended = set()
    for u, rated in test_ratings.items():
        top = recommend(score_fn, u, k)
        recommended |= set(top)
        out["novelty"].append(np.mean(-np.log2(p_item[top])))           # self-information: item càng hiếm càng "mới lạ"
        out["diversity"].append(len(set(genre[top])) / len(set(genre)))  # tỉ lệ thể loại có mặt trong top-k
        liked = [i for i, r in rated.items() if r >= 4]
        if liked:
            out["precision@10"].append(precision_at_k(top, liked, k))
            out["NDCG@10"].append(ndcg_at_k(top, {i: r - 3 for i, r in rated.items() if r >= 4}, k))
    res = {m: float(np.mean(v)) for m, v in out.items()}
    res["coverage"] = len(recommended) / n_items                        # tỉ lệ danh mục từng được gợi ý
    return res


def rmse(pred):
    return float(np.sqrt(np.mean((y_te - np.clip(pred, 1, 5)) ** 2)))

In [ ]:
# baseline predictor + ALS trên phần dư (như bài 03)
bu, bi = np.zeros(n_users), np.zeros(n_items)
for _ in range(15):
    bi = np.bincount(i_tr, y_tr - mu - bu[u_tr], n_items) / (5 + np.bincount(i_tr, minlength=n_items))
    bu = np.bincount(u_tr, y_tr - mu - bi[i_tr], n_users) / (5 + np.bincount(u_tr, minlength=n_users))
resid = y_tr - (mu + bu[u_tr] + bi[i_tr])
k_f, lam = 10, 5.0
rng = np.random.default_rng(0)
P, Q = rng.normal(0, 0.1, (n_users, k_f)), rng.normal(0, 0.1, (n_items, k_f))
by_user = [np.flatnonzero(u_tr == u) for u in range(n_users)]
by_item = [np.flatnonzero(i_tr == i) for i in range(n_items)]
for _ in range(15):
    for u, idx in enumerate(by_user):
        P[u] = np.linalg.solve(Q[i_tr[idx]].T @ Q[i_tr[idx]] + lam * np.eye(k_f), Q[i_tr[idx]].T @ resid[idx])
    for i, idx in enumerate(by_item):
        Q[i] = np.linalg.solve(P[u_tr[idx]].T @ P[u_tr[idx]] + lam * np.eye(k_f), P[u_tr[idx]].T @ resid[idx])

U_b, s_b, Vt_b = np.linalg.svd(B, full_matrices=False)
B5 = (U_b[:, :5] * s_b[:5]) @ Vt_b[:5]
norm_u = np.sqrt(B.sum(1)) + 1e-9
S_user = (B @ B.T) / np.outer(norm_u, norm_u)
np.fill_diagonal(S_user, 0)

models = {
    "ngẫu nhiên": (None, lambda u: np.random.default_rng(u).random(n_items)),
    "popularity": (None, lambda u: popularity),
    "baseline μ + b_u + b_i": (mu + bu[u_te] + bi[i_te], lambda u: mu + bu[u] + bi),
    "MF dự đoán rating (ALS)": (mu + bu[u_te] + bi[i_te] + np.sum(P[u_te] * Q[i_te], 1), lambda u: mu + bu[u] + bi + Q @ P[u]),
    "user-KNN (ngầm)": (None, lambda u: S_user[u, np.argsort(-S_user[u])[:30]] @ B[np.argsort(-S_user[u])[:30]]),
    "PureSVD k = 5 (ngầm)": (None, lambda u: B5[u]),
}
table = pd.DataFrame({name: {"RMSE": (rmse(pred) if pred is not None else np.nan), **evaluate(fn)}
                      for name, (pred, fn) in models.items()}).T
table[["RMSE", "precision@10", "NDCG@10"]].round(4)

Mô hình có **RMSE tốt nhất** (MF dự đoán rating) lại xếp hạng **kém xa** popularity và các mô hình học từ phản hồi ngầm. Vì sao?

- Test chỉ chứa phim user **đã chọn xem**. Việc *chọn xem gì* phụ thuộc mạnh vào độ phổ biến và gu → dữ liệu thiếu **không ngẫu nhiên** (*missing not at random*, MNAR).
- MF tối ưu rating **trên các phim đã xem**; khi xếp hạng **mọi** phim, nó đẩy lên các phim ít người xem có rating dự đoán cao — những phim user gần như chắc chắn sẽ không chọn.
- Mô hình học từ phản hồi ngầm (đã xem / chưa xem) học đúng câu hỏi "user sẽ chọn gì".

> Bài học: **tối ưu metric khớp với cách sản phẩm dùng mô hình**. Gợi ý top-N → đánh giá bằng metric xếp hạng; RMSE chỉ phù hợp khi thật sự cần dự đoán điểm (vd hiển thị "bạn sẽ chấm phim này 4.2★").

## 3. Thiên lệch chọn lựa: rating quan sát được không đại diện

Bộ dữ liệu được sinh với giả định user **tự chọn** phim hợp gu (`selection_strength`). So sánh với kịch bản user xem phim ngẫu nhiên (chỉ theo độ phổ biến):

In [ ]:
for strength in [0.0, 1.0, 2.0]:
    r_sim, _ = make_ratings(seed=42, selection_strength=strength)
    print(f"selection_strength = {strength}: rating trung bình quan sát được = {r_sim.rating.mean():.2f} | tỉ lệ 5★ = {np.mean(r_sim.rating == 5):.0%}")

Cùng một "gu thật", nhưng khi user tự chọn phim, rating **quan sát được** cao hơn hẳn — mô hình học trên dữ liệu này đánh giá quá cao những phim user sẽ không tự chọn. Trong thực tế: rating trên các nền tảng lệch mạnh về 4–5★; người ta chỉ review khi rất thích hoặc rất ghét. Hướng xử lý: mô hình hoá xác suất quan sát (propensity), **inverse propensity scoring** khi đánh giá, thu thập dữ liệu ngẫu nhiên hoá (một phần nhỏ gợi ý ngẫu nhiên).

## 4. Ngoài độ chính xác

| Metric | Định nghĩa | Vì sao quan trọng |
|---|---|---|
| **Coverage** | tỉ lệ danh mục từng xuất hiện trong gợi ý của ai đó | nhà cung cấp/nội dung đuôi dài có cơ hội không? |
| **Novelty** | trung bình $-\log_2 p(i)$ của item được gợi ý | gợi ý thứ user chưa biết, không chỉ hit phổ biến |
| **Diversity** | tỉ lệ thể loại khác nhau trong top-10 | tránh danh sách đơn điệu |

In [ ]:
table[["precision@10", "coverage", "novelty", "diversity"]].round(3)

- **Popularity** và **baseline** $\mu + b_u + b_i$ (thực chất cũng không cá nhân hoá: mọi user được xếp theo cùng $b_i$): độ phủ thấp nhất — mọi người thấy gần như cùng một danh sách.
- **Ngẫu nhiên**: độ phủ và độ mới lạ tối đa, vô dụng về độ chính xác.
- **MF dự đoán rating**: độ mới lạ cao (gợi ý phim hiếm) — nhưng đó là phim user không chọn.
- Mô hình ngầm (user-KNN, PureSVD): chính xác nhất, độ phủ cao hơn popularity nhiều; nhưng PureSVD có **độ đa dạng thấp nhất** (top-10 thường dồn vào vài thể loại). Không có mô hình nào thắng mọi tiêu chí → chọn theo mục tiêu sản phẩm.

## 5. Re-ranking đa dạng hoá: MMR

**Maximal Marginal Relevance**: chọn tham lam từ top-50 ứng viên; mỗi bước chọn item tối đa
$$\lambda\cdot\text{relevance}(i) - (1 - \lambda)\cdot\max_{j \in \text{đã chọn}}\text{sim}(i, j)$$
— cân bằng giữa phù hợp và khác biệt với những gì đã chọn (ở đây: sim = 1 nếu cùng thể loại).

In [ ]:
def mmr_scores_fn(base_scores_fn, lam_mmr, k=10, pool=50):
    def fn(u):
        scores = np.asarray(base_scores_fn(u), float).copy()
        scores[list(seen.get(u, []))] = -np.inf
        cand = list(np.argsort(-scores)[:pool])
        rel = (scores[cand] - scores[cand].min()) / (np.ptp(scores[cand]) + 1e-9)
        chosen = []
        while len(chosen) < k:
            best, best_val = None, -np.inf
            for c, r in zip(cand, rel):
                if c in chosen:
                    continue
                penalty = max((genre[c] == genre[j]) * 1.0 for j in chosen) if chosen else 0.0
                val = lam_mmr * r - (1 - lam_mmr) * penalty
                if val > best_val:
                    best, best_val = c, val
            chosen.append(best)
        out = np.full(n_items, -1e9)
        out[chosen] = np.arange(k, 0, -1)                               # điểm giả theo thứ tự đã chọn
        return out
    return fn


rows = {}
for lam_mmr in [1.0, 0.8, 0.6, 0.4]:
    rows[f"PureSVD + MMR λ = {lam_mmr}"] = evaluate(mmr_scores_fn(lambda u: B5[u], lam_mmr))
pd.DataFrame(rows).T[["precision@10", "NDCG@10", "diversity", "coverage"]].round(4)

In [ ]:
mmr = pd.DataFrame(rows).T
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(mmr["diversity"], mmr["precision@10"], "o-")
for name, row in mmr.iterrows():
    ax.annotate(name.split("MMR ")[1], (row["diversity"], row["precision@10"]), textcoords="offset points", xytext=(6, 6))
ax.set_xlabel("độ đa dạng của danh sách gợi ý")
ax.set_ylabel("precision@10")
ax.set_title("MMR: đổi một chút độ chính xác lấy nhiều độ đa dạng")
plt.show()

Giảm $\lambda$ → danh sách đa dạng hơn, độ chính xác offline giảm. Ở đây cái giá **rất nhỏ**: độ đa dạng tăng từ ~0.81 lên ~0.96 trong khi precision chỉ giảm khoảng 1% — vì trong 50 ứng viên đầu có nhiều phim gần như "tốt ngang nhau". Mức đánh đổi phù hợp chọn theo sản phẩm và nên kiểm chứng bằng A/B test (người dùng thật thường thích danh sách đa dạng hơn so với điều metric offline gợi ý).

## 6. Offline vs online

| Offline (dữ liệu lịch sử) | Online (A/B test, interleaving) |
|---|---|
| rẻ, nhanh, lặp lại được | đo hành vi thật: click, thời gian xem, doanh thu, giữ chân |
| thiên lệch bởi hệ thống cũ (MNAR) | tốn thời gian, rủi ro trải nghiệm |
| không đo được tác động lâu dài | đo được (nếu chạy đủ lâu) |

**Vòng phản hồi**: hệ thống gợi ý quyết định user thấy gì → user chỉ tương tác với thứ được thấy → dữ liệu train lần sau càng nghiêng về những gì hệ thống đã gợi ý (thiên lệch phổ biến tự khuếch đại). Giải pháp: khám phá (exploration — bandit, chương 35), theo dõi độ phủ/độ đa dạng theo thời gian.

## 7. Template: hệ gợi ý

| Bước | Câu hỏi | Lựa chọn |
|---|---|---|
| Bài toán | top-N hay dự đoán rating? phản hồi tường minh hay ngầm? | |
| Baseline | popularity (đếm, theo thời gian gần) | luôn có |
| Ứng viên (retrieval) | lấy vài trăm ứng viên từ hàng triệu item | item-KNN, MF/two-tower + ANN |
| Xếp hạng (ranking) | sắp xếp ứng viên với nhiều feature | boosting/learning-to-rank, mô hình sâu |
| Hậu xử lý | đa dạng, lọc đã xem, luật kinh doanh | MMR, quota |
| Đánh giá | offline: NDCG/recall@k theo thời gian, coverage; online: A/B | |
| Cold start | user/item mới | popularity, content-based, hỏi sở thích |

## 🧠 Kiểm tra nhanh

1. Tính precision@3, recall@3, MRR cho gợi ý `[X, Y, Z]` khi user thích `{Z, W}`.
2. Vì sao NDCG tốt hơn precision khi thứ tự quan trọng?
3. Vì sao mô hình RMSE thấp nhất lại xếp hạng kém? MNAR là gì?
4. Coverage, novelty, diversity đo gì? Vì sao popularity có coverage thấp?
5. MMR đánh đổi những gì? Vì sao cần A/B test để quyết định?

## 📌 Tóm tắt

- Metric xếp hạng: precision/recall@k, hit rate, MRR, MAP, NDCG (nhạy thứ tự, hỗ trợ mức liên quan).
- RMSE tốt ≠ gợi ý tốt: dữ liệu thiếu không ngẫu nhiên; học đúng câu hỏi (phản hồi ngầm cho top-N).
- Đo cả coverage, novelty, diversity; re-ranking MMR; offline không thay được online; cảnh giác vòng phản hồi.

## 📚 Tài liệu tham khảo

- Järvelin & Kekäläinen (2002) — *Cumulated Gain-based Evaluation of IR Techniques* (NDCG)
- Marlin & Zemel (2009) — *Collaborative Prediction and Ranking with Non-Random Missing Data*
- Carbonell & Goldstein (1998) — *The Use of MMR, Diversity-Based Reranking*
- Schnabel et al. (2016) — *Recommendations as Treatments: Debiasing Learning and Evaluation*